### A. Implementing attention mechanisms in an NLP model using TensorFlow/Keras.

In [2]:
import tensorflow as tf
from tensorflow.keras import layers, Model

#  Hyperparameters (tweak for your task) 
SRC_VOCAB = 8000
TGT_VOCAB = 8000
EMB_DIM    = 256
UNITS      = 256         # GRU hidden size
SRC_MAXLEN = 50
TGT_MAXLEN = 50

#  Encoder 
encoder_inputs = layers.Input(shape=(SRC_MAXLEN,), name="encoder_inputs")
enc_emb = layers.Embedding(SRC_VOCAB, EMB_DIM, mask_zero=True, name="enc_embedding")(encoder_inputs)
enc_out, enc_state = layers.GRU(UNITS, return_sequences=True, return_state=True, name="enc_gru")(enc_emb)
# enc_out: (B, SrcLen, UNITS)  | enc_state: (B, UNITS)

#  Decoder (teacher forcing) 
decoder_inputs = layers.Input(shape=(TGT_MAXLEN,), name="decoder_inputs")
dec_emb = layers.Embedding(TGT_VOCAB, EMB_DIM, mask_zero=True, name="dec_embedding")(decoder_inputs)
dec_seq, _ = layers.GRU(UNITS, return_sequences=True, return_state=True, name="dec_gru")(dec_emb, initial_state=enc_state)
# dec_seq: (B, TgtLen, UNITS)  — this is the query for attention at each step

#  Bahdanau / Additive Attention 
# Query: dec_seq (B, Tq, UNITS)
# Value/Key: enc_out (B, Tv, UNITS)
attn = layers.AdditiveAttention(name="bahdanau_attention")
context = attn([dec_seq, enc_out])              # (B, Tq, UNITS)
# Optional: expose attention weights via a custom call if needed in an interview.

#  Concatenate context with decoder sequence and predict tokens 
x = layers.Concatenate(name="concat_context")([dec_seq, context])   # (B, Tq, 2*UNITS)
x = layers.TimeDistributed(layers.Dense(UNITS, activation="tanh"), name="fusion_dense")(x)
logits = layers.TimeDistributed(layers.Dense(TGT_VOCAB), name="token_logits")(x)  # from_logits=True

model = Model([encoder_inputs, decoder_inputs], logits, name="seq2seq_bahdanau")
model.compile(optimizer="adam",
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=["accuracy"])

model.summary()

Model: "seq2seq_bahdanau"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ decoder_inputs      │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_inputs      │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, 50)        │          0 │ decoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_embedding       │ (None, 50, 256)   │  2,048,000 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 50)        │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convert_to_tensor   │ (None, 50)        │          0 │ not_equal_1[0][0] │
│ (ConvertToTensor)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_embedding       │ (None, 50, 256)   │  2,048,000 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_gru (GRU)       │ [(None, 50, 256), │    394,752 │ enc_embedding[0]… │
│                     │ (None, 256)]      │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_dims         │ (None, 50, 1)     │          0 │ not_equal_1[0][0] │
│ (ExpandDims)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_dims_1       │ (None, 50, 1)     │          0 │ convert_to_tenso… │
│ (ExpandDims)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_gru (GRU)       │ [(None, 50, 256), │    394,752 │ dec_embedding[0]… │
│                     │ (None, 256)]      │            │ enc_gru[0][1]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ broadcast_to        │ (None, 50, 256)   │          0 │ expand_dims[0][0] │
│ (BroadcastTo)       │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ broadcast_to_1      │ (None, 50, 256)   │          0 │ expand_dims_1[0]… │
│ (BroadcastTo)       │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bahdanau_attention  │ (None, 50, 256)   │        256 │ dec_gru[0][0],    │
│ (AdditiveAttention) │                   │            │ enc_gru[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 50, 512)   │          0 │ broadcast_to[0][… │
│ (Concatenate)       │                   │            │ broadcast_to_1[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_context      │ (None, 50, 512)   │          0 │ dec_gru[0][0],    │
│ (Concatenate)       │                   │            │ bahdanau_attenti… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ any (Any)           │ (None, 50)        │          0 │ concatenate[0][0

 Total params: 7,073,088 (26.98 MB)

 Trainable params: 7,073,088 (26.98 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

#  Hyperparameters 
SRC_VOCAB = 8000
TGT_VOCAB = 8000
EMB_DIM    = 256
UNITS      = 256         # GRU hidden size
SRC_MAXLEN = 50
TGT_MAXLEN = 50

# Encoder
encoder_inputs = layers.Input(shape=(SRC_MAXLEN,), name="encoder_inputs")
enc_emb = layers.Embedding(SRC_VOCAB, EMB_DIM, mask_zero=True, name="enc_embedding")(encoder_inputs)
enc_out, enc_state = layers.GRU(UNITS, return_sequences=True, return_state=True, name="enc_gru")(enc_emb)
# enc_out: (B, SrcLen, UNITS)  | enc_state: (B, UNITS)

# Decoder 
decoder_inputs = layers.Input(shape=(TGT_MAXLEN,), name="decoder_inputs")
dec_emb = layers.Embedding(TGT_VOCAB, EMB_DIM, mask_zero=True, name="dec_embedding")(decoder_inputs)
dec_seq, _ = layers.GRU(UNITS, return_sequences=True, return_state=True, name="dec_gru")(dec_emb, initial_state=enc_state)
# dec_seq: (B, TgtLen, UNITS)  — this is the query for attention at each step

# Bahdanau / Additive Attention
# Query: dec_seq (B, Tq, UNITS)
# Value/Key: enc_out (B, Tv, UNITS)
attn = layers.AdditiveAttention(name="bahdanau_attention")
context = attn([dec_seq, enc_out])              # (B, Tq, UNITS)
# Optional: expose attention weights via a custom call if needed in an interview.

#  Concatenate context with decoder sequence and predict tokens 
x = layers.Concatenate(name="concat_context")([dec_seq, context])   # (B, Tq, 2*UNITS)
x = layers.TimeDistributed(layers.Dense(UNITS, activation="tanh"), name="fusion_dense")(x)
logits = layers.TimeDistributed(layers.Dense(TGT_VOCAB), name="token_logits")(x)  # from_logits=True

model = Model([encoder_inputs, decoder_inputs], logits, name="seq2seq_bahdanau")
model.compile(optimizer="adam",
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=["accuracy"])

model.summary()

In [ ]:
class LuongAttention(layers.Layer):
    """
    attn_type: "dot" or "general"
    query:  (batch, d_q)        -> decoder hidden state
    values: (batch, T, d_v)     -> encoder outputs
    mask:   (batch, T) or None
    returns: context (batch, d_v), attn_weights (batch, T)
    """
    def __init__(self, attn_type="dot"):
        super().__init__()
        assert attn_type in ("dot", "general")
        self.attn_type = attn_type
        self.Wa = None  # created lazily for "general"

    def build(self, input_shapes):
        # no explicit build from Keras call signature, so defer to first call
        pass

    def _ensure_Wa(self, values, query):
        if self.attn_type == "general" and self.Wa is None:
            d_q = query.shape[-1]
            self.Wa = layers.Dense(d_q, use_bias=False)  # project values to query dim

    def call(self, query, values, mask=None):
        # query: (batch, d_q), values: (batch, T, d_v)
        self._ensure_Wa(values, query)

        if self.attn_type == "dot":
            # scores_t = v_t · q
            scores = tf.reduce_sum(values * tf.expand_dims(query, 1), axis=-1)      # (batch, T)
        else:  # "general"
            v_proj = self.Wa(values)                                                # (batch, T, d_q)
            scores = tf.reduce_sum(v_proj * tf.expand_dims(query, 1), axis=-1)      # (batch, T)

        if mask is not None:
            minus_inf = tf.constant(-1e9, dtype=scores.dtype)
            scores = tf.where(tf.equal(mask, 1), scores, minus_inf)

        attn_weights = tf.nn.softmax(scores, axis=-1)                                # (batch, T)
        context = tf.reduce_sum(tf.expand_dims(attn_weights, -1) * values, axis=1)   # (batch, d_v)
        return context, attn_weights

### B. Practical session: Building a simple Transformer model from scratch.

In [3]:
import tensorflow as tf 
from tensorflow import keras 
from tensorflow.keras import layers 

# Hyperparameters 
VOCAB_SIZE = 20000
MAX_LEN = 200
EMBED_DIM = 64 
NUM_HEADS = 4
FF_DIM = 128 
NUM_ENCODER_BLOCKS = 2 
DROPOUT = 0.1
BATCH_SIZE = 256
EPOCHS = 1 

# Data 
(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=VOCAB_SIZE)
x_train = keras.preprocessing.sequence.pad_sequences(x_train, maxlen=MAX_LEN, padding='post', truncating='post')
x_test = keras.preprocessing.sequence.pad_sequences(x_test, maxlen=MAX_LEN, padding='post', truncating='post')

# Positional Embedding Layer 
class PositionalEmbedding(layers.Layer):
    def __init__(self, vocab_size, embed_dim, max_len, **kwargs):
        super().__init__(**kwargs)
        self.token_emb = layers.Embedding(vocab_size, embed_dim, mask_zero=True)
        self.pos_emb = layers.Embedding(max_len, embed_dim)

    def compute_mask(self, inputs, mask=None):
        return self.token_emb.compute_mask(inputs)
    
    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[-1], delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions 

In [4]:
# Transformer Encoder Block
class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation='relu'),
            layers.Dense(embed_dim),
        ])
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)

    def call(self, x, mask=None, training=False):
        attn_mask = None 
        if mask is not None:
            attn_mask = tf.cast(mask[:, tf.newaxis, tf.newaxis, :],dtype=tf.bool)

        x1 = self.norm1(x)
        attn_out = self.attn(x1, x1, attention_mask=attn_mask, training=training)
        attn_out = self.drop1(attn_out, training = training )
        x2 = x+attn_out

        y1 = self.norm2(x2)
        ffn_out = self.ffn(y1)
        ffn_out = self.drop2(ffn_out, training = training)
        return x2 + ffn_out 

In [7]:
inputs = keras.Input(shape=(MAX_LEN, ), dtype='int32')
x = PositionalEmbedding(VOCAB_SIZE, EMBED_DIM, MAX_LEN)(inputs)
mask = x._keras_mask 

for _ in range(NUM_ENCODER_BLOCKS):
    x = TransformerEncoder(EMBED_DIM, NUM_HEADS, FF_DIM, DROPOUT)(x, mask=mask)

class MaskedWhere(layers.Layer):
    def call(self, inputs):
        x, mask, neg_inf = inputs
        mask_expanded = tf.expand_dims(mask, -1)
        return tf.where(mask_expanded, x, neg_inf)
    
if mask is not None:
    neg_inf = tf.cast(-1e9, x.dtype)
    masked_x = MaskedWhere()([x, mask, neg_inf])
    x = tf.reduce_max(masked_x, axis=1)
else: 
    x = layers.GlobalMaxPooling()(x)

x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

model = keras.Model(inputs, outputs)
model.compile(optimizer=keras.optimizers.Adam(learning_rate=2e-4), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

/Users/orbinsunny/.pyenv/versions/3.12.3/lib/python3.12/site-packages/keras/src/layers/layer.py:970: UserWarning: Layer 'transformer_encoder_4' (of type TransformerEncoder) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/Users/orbinsunny/.pyenv/versions/3.12.3/lib/python3.12/site-packages/keras/src/layers/layer.py:970: UserWarning: Layer 'transformer_encoder_5' (of type TransformerEncoder) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


ValueError: A KerasTensor cannot be used as input to a TensorFlow function. A KerasTensor is a symbolic placeholder for a shape and dtype, used when constructing Keras Functional models or Keras Functions. You can only use it as input to a Keras layer or a Keras operation (from the namespaces `keras.layers` and `keras.ops`). You are likely doing something like:

```
x = Input(...)
...
tf_fn(x)  # Invalid.
```

What you should do instead is wrap `tf_fn` in a layer:

```
class MyLayer(Layer):
    def call(self, x):
        return tf_fn(x)

x = MyLayer()(x)
```


### C. Implementing a pre-trained Transformer model for text classification.

### D.  Fine-tuning a pre-trained Transformer model on a custom dataset.

In [11]:
# Required imports
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    TrainingArguments, 
    Trainer
)
from sklearn.model_selection import train_test_split
import pandas as pd

# 1. Load custom dataset (example: sentiment analysis)
# Assuming CSV with 'text' and 'label' columns
data = pd.read_csv('custom_dataset.csv')

# Split data
train_texts, val_texts, train_labels, val_labels = train_test_split(
    data['text'].tolist(), 
    data['label'].tolist(), 
    test_size=0.2,
    random_state=42
)

print(f"Training samples: {len(train_texts)}")
print(f"Validation samples: {len(val_texts)}")

/Users/orbinsunny/.pyenv/versions/3.12.3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ValueError: Your currently installed version of Keras is Keras 3, but this is not yet supported in Transformers. Please install the backwards-compatible tf-keras package with `pip install tf-keras`.

### E. Implementing optimization techniques in Transformer models.

## Practice

In [5]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

vocab_size = 20000
max_len = 200

(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=vocab_size)
x_train = keras.preprocessing.sequence.pad_sequences(x_train, maxlen=max_len, padding="post")
x_test  = keras.preprocessing.sequence.pad_sequences(x_test,  maxlen=max_len, padding="post")

class PositionalEncoding(layers.Layer):
    def __init__(self, max_len, d_model):
        super().__init__()
        pos = tf.range(max_len)[:, tf.newaxis]
        i = tf.range(d_model)[tf.newaxis, :]
        angle_rates = 1 / tf.pow(10000.0, (2*(i//2))/tf.cast(d_model, tf.float32))
        angles = tf.cast(pos, tf.float32) * angle_rates
        sines = tf.math.sin(angles[:, 0::2])
        coses = tf.math.cos(angles[:, 1::2])
        pe = tf.zeros_like(angles)
        pe = tf.tensor_scatter_nd_update(pe, tf.where(tf.range(d_model) % 2 == 0)[:, None], tf.reshape(sines, [-1]))
        pe = tf.tensor_scatter_nd_update(pe, tf.where(tf.range(d_model) % 2 == 1)[:, None], tf.reshape(coses, [-1]))
        self.pe = pe[tf.newaxis, ...]

    def call(self, x):
        seq_len = tf.shape(x)[1]
        return x + self.pe[:, :seq_len, :]

class TransformerBlock(layers.Layer):
    def __init__(self, d_model, num_heads, mlp_dim, dropout=0.1):
        super().__init__()
        self.mha = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model, dropout=dropout)
        self.ffn = keras.Sequential([
            layers.Dense(mlp_dim, activation="relu"),
            layers.Dense(d_model),
        ])
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)

    def call(self, x, padding_mask=None, training=False):
        if padding_mask is not None:
            attn_mask = keras.ops.cast(padding_mask, "bool")
        else:
            attn_mask = None

        attn_output = self.mha(query=x, value=x, key=x, attention_mask=attn_mask, training=training)
        x = self.norm1(x + self.drop1(attn_output, training=training))
        ffn_output = self.ffn(x, training=training)
        x = self.norm2(x + self.drop2(ffn_output, training=training))
        return x

d_model   = 128
num_heads = 4
mlp_dim   = 256
dropout   = 0.1

inputs = keras.Input(shape=(max_len,), dtype="int32")
mask = layers.Lambda(lambda t: keras.ops.cast(keras.ops.not_equal(t, 0), "float32"))(inputs)
x = layers.Embedding(input_dim=vocab_size, output_dim=d_model)(inputs)
x = PositionalEncoding(max_len=max_len, d_model=d_model)(x)
x = TransformerBlock(d_model, num_heads, mlp_dim, dropout)(x, padding_mask=mask)

mask_expanded = tf.expand_dims(mask, axis=-1)
sum_embeddings = tf.reduce_sum(x * mask_expanded, axis=1)
lengths = tf.reduce_sum(mask, axis=1, keepdims=True) + 1e-9
pooled = sum_embeddings / lengths

x = layers.Dropout(0.2)(pooled)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs)
model.compile(optimizer=keras.optimizers.Adam(3e-4),
              loss="binary_crossentropy",
              metrics=["accuracy"])

model.summary()

history = model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=2,
    batch_size=256,
    verbose=1
)

test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.3f}")

TypeError: `x` and `y` must have the same dtype, got tf.int32 != tf.float32.

In [8]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

# ---------------------------------
# 1. OUR SIMPLE TEST DATA
# ---------------------------------
test_data_text = ["how are you", 
                  "i am fine thank you", 
                  "are you okay"]

# ---------------------------------
# 2. AUTO-TOKENIZE & PAD DATA
# ---------------------------------
max_len = 10     # Max words per sentence
vocab_size = 20  # Max words in our vocabulary

vectorizer = layers.TextVectorization(
    max_tokens=vocab_size, 
    output_sequence_length=max_len
)
vectorizer.adapt(test_data_text)

tokenized_data = vectorizer(test_data_text)

print("--- 1. Tokenized & Padded Data ---")
print("Vocabulary:", vectorizer.get_vocabulary())
print("Tokenized data (shape):", tokenized_data.shape)
print(tokenized_data.numpy())
print("-" * 30)

# ---------------------------------
# 3. THE TRANSFORMER BLOCK
# (This class is correct and unchanged)
# ---------------------------------
class TransformerBlock(layers.Layer):
    def __init__(self, d_model, num_heads, mlp_dim, dropout=0.1):
        super().__init__()
        self.mha = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model, dropout=dropout)
        self.ffn = keras.Sequential([
            layers.Dense(mlp_dim, activation="relu"),
            layers.Dense(d_model),
        ])
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)

    def call(self, x, padding_mask=None, training=False):
        if padding_mask is not None:
            # MHA expects mask shape (batch, 1, seq_len)
            attn_mask = tf.cast(padding_mask[:, tf.newaxis, :], tf.bool)
        else:
            attn_mask = None
            
        attn_output = self.mha(query=x, value=x, key=x, attention_mask=attn_mask, training=training)
        x = self.norm1(x + self.drop1(attn_output, training=training)) 

        ffn_output = self.ffn(x, training=training)
        x = self.norm2(x + self.drop2(ffn_output, training=training))
        return x

# ---------------------------------
# 4. *** FIX #2: Custom Pooling Layer ***
# All the pooling logic must also be in a layer.
# ---------------------------------
class MaskedGlobalAveragePooling(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def call(self, inputs):
        # Inputs will be a list: [embeddings, mask]
        x, mask = inputs
        
        mask_expanded = tf.expand_dims(mask, axis=-1)
        sum_embeddings = tf.reduce_sum(x * mask_expanded, axis=1)
        lengths = tf.reduce_sum(mask, axis=1, keepdims=True) + 1e-9
        pooled_output = sum_embeddings / lengths
        return pooled_output

# ---------------------------------
# 5. BUILD THE MODEL (FUNCTIONAL API)
# ---------------------------------
# Model Hyperparameters
d_model = 32     
num_heads = 4    
mlp_dim = 64     

# --- Step 1: Input ---
inputs = keras.Input(shape=(max_len,), dtype="int32")

# --- Step 2: Create the Padding Mask (*** FIX #1 ***)
# We MUST wrap the tf.not_equal logic in a Keras layer.
mask = layers.Lambda(
    lambda x: tf.cast(tf.not_equal(x, 0), tf.float32),
    name="create_mask"
)(inputs)

# --- Step 3: Token Embeddings ---
token_embeddings_layer = layers.Embedding(
    input_dim=vectorizer.vocabulary_size(), 
    output_dim=d_model
)
x = token_embeddings_layer(inputs)

# --- Step 4: Add Positional Embeddings (The *Simple* Way) ---
positional_embeddings_layer = layers.Embedding(
    input_dim=max_len, 
    output_dim=d_model
)
positions = tf.range(start=0, limit=max_len, delta=1)
pos_embeddings = positional_embeddings_layer(positions)

x = x + pos_embeddings
# 

# --- Step 5: The Transformer Block ---
transformer = TransformerBlock(d_model, num_heads, mlp_dim)
# We pass the *tensor* 'x' and the *tensor* 'mask'
x = transformer(x, padding_mask=mask)
# 

# --- Step 6: Pooling (*** FIX #2 ***) ---
# We pass BOTH the embeddings and the mask to our new custom layer.
# The input is a list: [embeddings_tensor, mask_tensor]
pooled_output = MaskedGlobalAveragePooling()([x, mask])

# --- Step 7: Create and Run the Model ---
model = keras.Model(inputs=inputs, outputs=pooled_output)

# Pass our simple tokenized data through the model
final_sentence_embeddings = model(tokenized_data)

print("\n--- 2. Final Output ---")
print("Final 'sentence embedding' shape:", final_sentence_embeddings.shape)
print("Final 'sentence embeddings' (one vector per sentence):")
print(final_sentence_embeddings.numpy())

--- 1. Tokenized & Padded Data ---
Vocabulary: ['', '[UNK]', np.str_('you'), np.str_('are'), np.str_('thank'), np.str_('okay'), np.str_('i'), np.str_('how'), np.str_('fine'), np.str_('am')]
Tokenized data (shape): (3, 10)
[[7 3 2 0 0 0 0 0 0 0]
 [6 9 8 4 2 0 0 0 0 0]
 [3 2 5 0 0 0 0 0 0 0]]
------------------------------

--- 2. Final Output ---
Final 'sentence embedding' shape: (3, 32)
Final 'sentence embeddings' (one vector per sentence):
[[-0.6237757   0.9723902  -0.677506   -1.0966741  -0.19483094  0.1942391
  -0.5004926  -0.02406156  0.05605922 -0.47361603  0.4232396  -0.07742844
  -0.47981778  0.38070002 -0.9023326   0.15514706  0.27828565 -0.6786713
   0.21119939  1.1830276   1.0498668   0.06811688  0.5254845   0.7605583
  -0.296073    0.66263884 -0.2761457  -0.16003907  1.3878336  -0.4507463
  -0.85394216 -0.54263365]
 [-0.40682477  1.0490234  -0.58834404 -0.44444877 -0.11142068  0.35049433
   0.21564326 -0.29752353  0.6564086  -0.15023148  0.37313542 -0.01619549
  -0.10962303 

--- 1. Tokenized & Padded Data ---
Vocabulary: ['', '[UNK]', np.str_('you'), np.str_('are'), np.str_('thank'), np.str_('okay'), np.str_('i'), np.str_('how'), np.str_('fine'), np.str_('am')]
Tokenized data (shape): (3, 10)
[[7 3 2 0 0 0 0 0 0 0]
 [6 9 8 4 2 0 0 0 0 0]
 [3 2 5 0 0 0 0 0 0 0]]
------------------------------

--- 2. Final Output ---
Final 'sentence embedding' shape: (3, 32)
Final 'sentence embeddings' (one vector per sentence):
[[ 1.3240005   0.54069024  0.6915496   0.10092846 -0.7177562   0.17835124
  -0.22224212 -0.23329262 -0.14297204 -0.78249454  0.02852708  0.38714495
  -0.31293    -1.216376   -0.06621533  1.0552646   0.68949413  0.16504014
  -0.00764721 -0.6295748   0.08589641 -0.7672196   0.9237397  -0.92071253
  -0.43529478  0.52673125 -0.19191587 -0.9017999  -0.672471    0.18359132
   0.07028437  1.2696806 ]
 [ 0.87419397 -0.15833582  0.7034215  -0.3715481  -0.5077249   0.46309668
  -0.05084047 -0.11846845  0.3942297  -0.17565855 -0.36052775 -0.19848783
  -0.50138

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np



class TransformerBlock(layers.Layer):
    def __init__(self, d_model, num_heads, mlp_dim, dropout=0.1):
        self.mha = layers.MultiHeadAttention(num_heads = num_heads, key_dim = d_model, drop_out=dropout)
        self.ffn = keras.Sequential([
            layers.Dense(mlp_dim, activation='relu'),
            layers.Dense(d_model)
        ])
        self.norm1 = layers.LayerNormalisation(epsilon=1e-6)
        self.norm2 = layers.LayerNormalisaiotn(epsilon=1e-6)
        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)
    
    def call(self, x, padding_mask=None, training=False):
        # qkv
        attn_output = self.mha(query= x, value=x, key=x, attention_mask=padding_mask)

        # norm
        x = self.norm1(x + self.drop1(attn_output, training=training))
        ffn_ouput = self.ffn(x, training = training)

        x = self.norm2(x+self.drop2(ffn_ouput, training=training))

        return x
    
vocab_size = 10000
max_len = 50

(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=vocab_size)

x_train = keras.preprocessing.sequence.pad_sequences(x_train, maxlen=max_len, padding='post')
x_test = keras.preprocessing.sequence.pad_sequences(x_test, maxlen=max_len, padding='post')

d_model = 64
num_heads = 4
mlp_dim = 128

inputs = keras.Input(shape=(max_len, ), dtype='int32')

mask = layers.Lambda(lambda t: tf.cast(tf.not_equal(t, 0), tf.float32))(inputs)

x = layers.Embedding(input_dim=vocab_size, output_dim=d_model)(inputs)

pos_indices = tf.range(start=0, limit=max_len, delta=1)
pos_embedding_layer = layers.Embedding(input_dim_max_len, output_dim=d_model)
x = x+pos_embedding_layer(pos_indices)

x = TransformerBlock(d_model, num_heads, mlp_dim)(x, padding_mask=mask)

def masked_pool(inputs):
    x, mask = inputs 
    mask_expanded = tf.expand_dim(mask, axis=-1)
    sum_x =tf.reduce_sum(x*mask_expanded, axis=1)
    sum_mask = tf.reduce_sum(mask, axis=1, keepdims=True)+1e9
    return sum_x / sum_mask

pooled = layers.Lambda(masked_pool, name='masked_pooling')([x, mask])
outputs = layers.Dense(1, activation='sigmoid')(pooled)

model = keras


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

class TransformerBlock(layers.Layer):
    def __init__(self, d_model, num_heads, mlp_dim, dropout=0.1):
        self.mha = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model)
        self.ffn = keras.Sequential([
            layers.Dense(mlp_dim, activation='relu'),
            layers.Dense(d_model)
        ])
        self.norm1 = layers.LayerNomalisation()
        self.norm2 = layers.LayerNomalisation()
        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)

    def call(self, x, padding_mask=None, training=False):
        attn_output = self.mha(query=x, value=x, key=x, attention_mask=padding_mask)
        x = self.norm1(x + self.drop1(attn_output, training=training))
        ffn_output = self.ffn(x, training=training)
        x = self.norm2(x + self.drop2(ffn_output, training=training))
        return x

d_model = 64
num_heads =4
mlp_dim = 128

inputs = keras.Input(shape=(max_len, ), dtype= 'int32')

mask = layers.Lambda(lambda t: tf.cast(tf.not_equal(t, 0) , tf.float32))(inputs)

x = layers.Embedding(input_dim=vocab_size , output_dim=d_model)(inputs)

pos_indices = tf.range(start=0, limit=max_len, delta=1)
pos_embedding_layer = layers.Embedding(input_dim=max_len, output_dim = d_model)
x = x + pos_embedding_layer(pos_indices)

x = TransformerBlock(d_model, num_heads, mlp_dim)(x, padding_mask=mask)

def masked_pool(inputs):
    x, mask = inputs
    mask_expanded = tf.expand_dims(mask, axis=1)
    sum_x = tf.reduce_sum(x*mask_expanded, axis=1)
    sum_mask = tf.reduce_sum(mask, axis=1, keepdims=True) + 1e-9
    return sum_x/sum_mask 

pooled = layers.Lambda(masked_pool)